In [16]:
import numpy as np
import matplotlib.pyplot as plt

class KMeans:
    def __init__(self, k, initial_centroids=None, max_iterations=100):
        self.k = k
        self.initial_centroids = initial_centroids
        self.max_iterations = max_iterations
        self.centroids_history = []
        self.labels_history = []
        
    def fit(self, X):
        n_samples = X.shape[0]
        
        # Initialize centroids
        if self.initial_centroids is not None:
            self.centroids = np.array(self.initial_centroids, dtype=float)
        else:
            random_indices = np.random.choice(n_samples, self.k, replace=False)
            self.centroids = X[random_indices].copy()
        
        self.centroids_history.append(self.centroids.copy())
        
        # Iterate
        for iteration in range(self.max_iterations):
            # Assign clusters
            labels = self._assign_clusters(X)
            self.labels_history.append(labels)
            
            # Update centroids
            old_centroids = self.centroids.copy()
            for i in range(self.k):
                cluster_points = X[labels == i]
                if len(cluster_points) > 0:
                    self.centroids[i] = cluster_points.mean(axis=0)
            
            self.centroids_history.append(self.centroids.copy())
            
            # Check convergence
            shift = np.linalg.norm(self.centroids - old_centroids)
            if shift < 1e-4:
                print(f"Converged in {iteration + 1} iterations")
                break
        
        self.labels_ = labels
        return self
    
    def _assign_clusters(self, X):
        distances = np.array([[np.linalg.norm(x - c) for c in self.centroids] for x in X])
        return np.argmin(distances, axis=1)


class KMeansPlusPlus:
    def __init__(self, k, first_centroid=None, max_iterations=100):
        self.k = k
        self.first_centroid = first_centroid
        self.max_iterations = max_iterations
        self.centroids_history = []
        self.labels_history = []
        
    def fit(self, X):
        n_samples = X.shape[0]
        
        # Initialize first centroid
        if self.first_centroid is not None:
            centroids = [np.array(self.first_centroid, dtype=float)]
        else:
            centroids = [X[np.random.randint(n_samples)].copy()]
        
        # K-Means++ initialization for remaining centroids
        for _ in range(1, self.k):
            distances = np.array([min([np.linalg.norm(x - c)**2 for c in centroids]) for x in X])
            probabilities = distances / distances.sum()
            cumulative_probs = np.cumsum(probabilities)
            r = np.random.rand()
            
            for idx, prob in enumerate(cumulative_probs):
                if r < prob:
                    centroids.append(X[idx].copy())
                    break
        
        self.centroids = np.array(centroids)
        self.centroids_history.append(self.centroids.copy())
        
        # Standard K-Means iterations
        for iteration in range(self.max_iterations):
            labels = self._assign_clusters(X)
            self.labels_history.append(labels)
            
            old_centroids = self.centroids.copy()
            for i in range(self.k):
                self.centroids[i] = X[labels == i].mean(axis=0)
            
            self.centroids_history.append(self.centroids.copy())
            
            shift = np.linalg.norm(self.centroids - old_centroids)
            if shift < 1e-4:
                print(f"K-Means++ converged in {iteration + 1} iterations")
                break
        
        self.labels_ = labels
        return self
    
    def _assign_clusters(self, X):
        distances = np.array([[np.linalg.norm(x - c) for c in self.centroids] for x in X])
        return np.argmin(distances, axis=1)


def generate_data(n_samples=300, noise_level=0.5, random_state=42):
    """Generate customer data: Age vs Spending Score"""
    np.random.seed(random_state)
    
    cov = [[20 * (1 + noise_level), 5], [5, 40 * (1 + noise_level)]]
    
    # 4 customer segments
    c1 = np.random.multivariate_normal([25, 75], cov, n_samples//4)  # Young high spenders
    c2 = np.random.multivariate_normal([25, 25], cov, n_samples//4)  # Young low spenders
    c3 = np.random.multivariate_normal([55, 75], cov, n_samples//4)  # Old high spenders
    c4 = np.random.multivariate_normal([55, 25], cov, n_samples//4)  # Old low spenders
    
    data = np.vstack([c1, c2, c3, c4])
    data[:, 0] = np.clip(data[:, 0], 18, 70)  # Age: 18-70
    data[:, 1] = np.clip(data[:, 1], 0, 100)  # Spending: 0-100
    
    return data


def plot_data(X):
    """Visualize original data"""
    plt.figure(figsize=(8, 6))
    plt.scatter(X[:, 0], X[:, 1], c='steelblue', alpha=0.6, s=50, edgecolors='navy')
    plt.xlabel('Age', fontsize=12, fontweight='bold')
    plt.ylabel('Spending Score', fontsize=12, fontweight='bold')
    plt.title('Original Customer Data', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_iterations(X, model, title):
    """Visualize centroid shifts across iterations"""
    n_iterations = len(model.centroids_history) - 1
    colors = plt.cm.Set1(np.linspace(0, 1, model.k))
    
    # Calculate grid size
    n_cols = 3
    n_rows = (n_iterations + n_cols) // n_cols
    
    fig = plt.figure(figsize=(15, 5 * n_rows))
    
    for iter_num in range(n_iterations + 1):
        ax = plt.subplot(n_rows, n_cols, iter_num + 1)
        
        # Plot points
        if iter_num > 0:
            labels = model.labels_history[iter_num - 1]
            for i in range(model.k):
                mask = labels == i
                ax.scatter(X[mask, 0], X[mask, 1], c=[colors[i]], alpha=0.5, s=40)
        else:
            ax.scatter(X[:, 0], X[:, 1], c='lightgray', alpha=0.5, s=40)
        
        # Plot centroids
        centroids = model.centroids_history[iter_num]
        ax.scatter(centroids[:, 0], centroids[:, 1], 
                  c='black', marker='X', s=400, 
                  edgecolors='yellow', linewidths=3, zorder=5)
        
        # Show centroid movement
        if iter_num > 0:
            old_centroids = model.centroids_history[iter_num - 1]
            for i in range(model.k):
                ax.arrow(old_centroids[i, 0], old_centroids[i, 1],
                        centroids[i, 0] - old_centroids[i, 0],
                        centroids[i, 1] - old_centroids[i, 1],
                        head_width=2, head_length=1.5, fc='blue', ec='blue', 
                        linewidth=2, alpha=0.7, length_includes_head=True)
        
        # Label centroids
        for i, c in enumerate(centroids):
            ax.text(c[0], c[1], f'C{i+1}', ha='center', va='center',
                   fontsize=9, fontweight='bold', color='white')
        
        ax.set_xlabel('Age', fontweight='bold')
        ax.set_ylabel('Spending Score', fontweight='bold')
        ax.set_title(f'Iteration {iter_num}', fontweight='bold')
        ax.grid(True, alpha=0.3)
    
    plt.suptitle(title, fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.show()


